# YOLO Inference

Runs inference with the best trained YOLO model on unseen test images.

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
import glob
from ultralytics import YOLO

## Configuration

In [ ]:
WEIGHTS_PATH = "runs/detect/yolov11/weights/best.pt"
TEST_DIR = "../../data/unified/combined/test/images"
IMG_SIZE = 640
CONFIDENCE_THRESHOLD = 0.25
OUTPUT_DIR = "runs/detect/test_inference"

## Batch Inference

In [ ]:
def run_batch_inference(weights_path, images_dir, img_size,
                        conf_threshold, output_dir):
    """
    Runs inference on all images in a folder.
    Visual results (images with bounding boxes) are saved to the output folder.
    """
    print("  YOLO Inference on Test Images")

    if not os.path.exists(weights_path):
        print(f"Model not found: {weights_path}")
        print("Train the model first with train.py")
        return

    if not os.path.isdir(images_dir):
        print(f"Images folder not found: {images_dir}")
        return

    extensions = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
    images = []
    for ext in extensions:
        images.extend(glob.glob(os.path.join(images_dir, ext)))
    print(f"  Found {len(images)} test images")

    print(f"  Loading model: {weights_path}")
    model = YOLO(weights_path)

    print(f"  Resolution: {img_size}px | Min. confidence: {conf_threshold}")
    print(f"  Processing...\n")

    results = model.predict(
        source=images_dir,
        imgsz=img_size,
        conf=conf_threshold,
        save=True,
        save_txt=True,
        save_conf=True,
        name=output_dir,
        exist_ok=True,
    )

    total_detections = sum(len(r.boxes) for r in results)
    print(f"\n Inference complete!")
    print(f" Images processed: {len(results)}")
    print(f" Total detections: {total_detections}")
    print(f" Visual results:   {output_dir}/")

    return results

## Single Image Inference

In [ ]:
def run_single_inference(weights_path, image_path, img_size, conf_threshold):
    """
    Runs inference on a single image and prints detection details.
    Useful for quick tests or demonstrations.
    """
    if not os.path.exists(weights_path):
        print(f" Model not found: {weights_path}")
        return
    if not os.path.exists(image_path):
        print(f" Image not found: {image_path}")
        return

    model = YOLO(weights_path)
    results = model.predict(
        source=image_path,
        imgsz=img_size,
        conf=conf_threshold,
        save=True,
        name="single_inference",
        exist_ok=True,
    )

    result = results[0]
    print(f"\n Detections in: {os.path.basename(image_path)}")
    print(f"Total: {len(result.boxes)} objects detected\n")

    for i, box in enumerate(result.boxes):
        cls_name   = result.names[int(box.cls)]
        confidence = float(box.conf)
        coords     = box.xyxy[0].tolist()
        print(f"   {i+1}. {cls_name:<15} confidence: {confidence:.0%}  "
              f"bbox: [{coords[0]:.0f}, {coords[1]:.0f}, "
              f"{coords[2]:.0f}, {coords[3]:.0f}]")

    return results

## Run

In [ ]:
run_batch_inference(
    weights_path=WEIGHTS_PATH,
    images_dir=TEST_DIR,
    img_size=IMG_SIZE,
    conf_threshold=CONFIDENCE_THRESHOLD,
    output_dir=OUTPUT_DIR,
)

In [ ]:
# run_single_inference(
#     weights_path=WEIGHTS_PATH,
#     image_path="/content/drive/MyDrive/dataset_split/test/images/example.png",
#     img_size=IMG_SIZE,
#     conf_threshold=CONFIDENCE_THRESHOLD,
# )